# Gravitational lensing by a black hole

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/gravitational_lensing)

Light rays past a black hole are null geodesics, traced for a whole fan of impact parameters at once: far rays bend by $4GM/(c^2 b)$, rays inside $\sqrt{27}\,GM/c^2$ fall in. On the sky, the lens equation maps every pixel back to the source plane in one tensor operation, so a star passing behind the hole turns into two arcs, an Einstein ring and a magnification spike.

**Vector form**

$$\frac{d^2u}{d\varphi^2} + u = \frac{3GM}{c^2}\,u^2,\qquad \alpha \approx \frac{4GM}{c^2 b},\qquad \boldsymbol\beta = \boldsymbol\theta - \theta_E^2\,\frac{\boldsymbol\theta}{\lvert\boldsymbol\theta\rvert^2},\qquad \theta_E = \sqrt{\frac{4GM}{c^2}\frac{D_{ls}}{D_l D_s}}$$

**Index (tensor) form**

$$u_b'' = -u_b + 3M\,u_b^2,\qquad \beta_{pi} = \theta_{pi} - \frac{\theta_{pi}}{\theta_{pj}\theta_{pj}},\qquad I_{fp} = S\big(\beta_{p} - c_{f}\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_b$ | `y[:, 0]` | (B,) | $1/r$ along ray $b$, starting at 0 (infinity) with $du/d\varphi = 1/b$ |
| $u_b''$ | `binet_rhs(y, M)` | (B, 2) | null-geodesic orbit equation, RK4 in $\varphi$ for all rays |
| $\alpha_b$ | `deflection_angles(rays)` | (B,) | exit angle minus $\pi$; NaN if captured |
| $\theta_{pi}$ | `sky_grid(n, 2.5)` | (n, n, 2) | sky angle of pixel $p$, in units of $\theta_E$ |
| $\theta_{pj}\theta_{pj}$ | `einsum('...j,...j->...', theta, theta)` | (n, n) | repeated $j$ ⇒ summed |
| $\beta_{pi}$ | `lens_map(theta)` | (n, n, 2) | where each pixel looks in the source plane |
| $I_{fp}$ | `source_brightness(beta[None], c[:, None, None])` | (F, n, n) | lensed image for every frame $f$ by broadcasting |
| $\mu_f$ | `lensed.sum((-2, -1)) / plain.sum((-2, -1))` | (F,) | magnification: total flux ratio |

Ray panel: units $G = c = M = 1$; the dotted circle is the photon sphere ($r = 3M$), and rays with $b < \sqrt{27}\,M \approx 5.2\,M$ end in the hole, which is why a black hole casts a shadow 2.6 times its horizon radius. Rays just outside can loop around before escaping. Sky panel: thin-lens point mass in units of the Einstein radius (about a milliarcsecond for a stellar black hole in our Galaxy, which is why microlensing is seen as a brightening rather than resolved images). Lensing conserves surface brightness, so the image is simply the source evaluated at $\beta(\theta)$; the extra light comes from the images being stretched. The finite source size caps the magnification near perfect alignment, where a point source would give $\mu \to \infty$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/gravitational_lensing/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    M = 1.0
    b, y0 = tensors.ray_fan(params["b_min"], params["b_max"], params["n_rays"], torch.device("cpu"))
    rays = solve.trace_rays(y0, M)
    alpha = solve.deflection_angles(rays)
    rays["paths"] = solve.ray_paths(rays)

    print(f"shadow: rays with b < √27 M = {equation.shadow_radius(M):.3f} M fall in")
    for b_k, a_k in zip(b.tolist(), alpha.tolist()):
        weak = equation.thin_lens_deflection(torch.tensor(b_k), M)
        fate = (
            "captured" if math.isnan(a_k) else f"α = {math.degrees(a_k):7.2f}°   4GM/(c²b) = {math.degrees(weak):6.2f}°"
        )
        print(f"b = {b_k:5.2f} M   {fate}")

    # a 10-solar-mass black hole halfway to a star 8 kpc away
    kpc = 3.0857e19  # m
    theta_E = equation.einstein_radius(10 * 1477.0, 4 * kpc, 8 * kpc)
    print(f"Einstein radius for 10 M_sun at 4 kpc, source at 8 kpc: {theta_E * 206264.806e3:.2f} milliarcseconds")

    half = 2.5
    theta = tensors.sky_grid(params["n_pixels"], half, dev)
    centres = tensors.source_path(120, params["impact"], 2.0, dev)
    lensed, plain = solve.lensed_images(theta, centres, params["source_radius"])
    mu = solve.magnification(lensed, plain)
    print(f"peak magnification {mu.max():.2f} (point source at u = {params['impact']:g}: "
          f"{equation.point_magnification(torch.tensor(max(params['impact'], 1e-3))):.2f})")  # fmt: skip

    sky = {"lensed": lensed, "centres": centres, "mu": mu, "half_width": half}
    return plot.animate(rays, b, alpha, M, sky, out_dir / "gravitational_lensing.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/gravitational_lensing/equation.py
r"""Physics → tensor translation.

Light passing a black hole of mass M follows a null geodesic. In the equatorial plane of the Schwarzschild metric
the orbit equation for u = 1/r as a function of the angle φ is (units G = c = 1)

    \frac{d^2 u_b}{dφ^2} = -u_b + 3M u_b^2

for a batch of rays b, each coming in from infinity with impact parameter b: u = 0, du/dφ = 1/b at φ = 0.
Without the 3Mu² term the ray is a straight line; with it, the ray is bent by α (it leaves at φ = π + α).
For b ≫ M the deflection is Einstein's α = 4GM/(c² b). Rays with b < \sqrt{27}\,M spiral in: that is the
black hole's shadow, the edge of which is the photon sphere at r = 3M.

Far from the hole the thin-lens approximation applies. Angles θ on the sky (in units of the Einstein radius
θ_E = \sqrt{4GM D_{ls} / (c^2 D_l D_s)}) map back to the source plane by the lens equation, for every pixel p:

    β_{pi} = θ_{pi} - \frac{θ_{pi}}{θ_{pj} θ_{pj}}      (j summed)

Surface brightness is conserved, so the lensed image is I(θ) = S(β(θ)); a source right behind the lens becomes an
Einstein ring, otherwise two images. A point source at offset u (in θ_E) is magnified by μ = (u² + 2)/(u\sqrt{u² + 4}).
"""

import math

import torch


def binet_rhs(y: torch.Tensor, M: float) -> torch.Tensor:
    """d/dφ of the state y (B, 2) = (u, du/dφ) for every ray."""
    u, du = y[:, 0], y[:, 1]
    return torch.stack([du, -u + 3 * M * u**2], dim=1)


def thin_lens_deflection(b: torch.Tensor, M: float) -> torch.Tensor:
    """Einstein's weak-field deflection α = 4GM/(c²b)."""
    return 4 * M / b


def shadow_radius(M: float) -> float:
    r"""Critical impact parameter \sqrt{27} GM/c²: rays closer than this are captured."""
    return math.sqrt(27) * M


def einstein_radius(GM_over_c2: float, D_l: float, D_s: float) -> float:
    r"""θ_E = \sqrt{4GM D_{ls} / (c^2 D_l D_s)} in radians (D_ls = D_s − D_l for flat space)."""
    return math.sqrt(4 * GM_over_c2 * (D_s - D_l) / (D_l * D_s))


def lens_map(theta: torch.Tensor) -> torch.Tensor:
    r"""β_{pi} = θ_{pi} − θ_{pi} / (θ_{pj} θ_{pj}) for any batch of sky positions (θ_E units).   (..., 2) → (..., 2)"""
    r2 = torch.einsum("...j,...j->...", theta, theta)[..., None]
    return theta - theta / r2


def source_brightness(beta: torch.Tensor, centre: torch.Tensor, radius: float) -> torch.Tensor:
    """A round source with a soft edge, centred at `centre` (..., 2) and evaluated at β (..., 2) → (...)."""
    d = (beta - centre).norm(dim=-1)
    return torch.sigmoid((radius - d) / (0.15 * radius))


def point_magnification(u: torch.Tensor) -> torch.Tensor:
    r"""μ = (u² + 2) / (u \sqrt{u² + 4}) for a point source at offset u (θ_E units), both images together."""
    return (u**2 + 2) / (u * torch.sqrt(u**2 + 4))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/gravitational_lensing/tensors.py
"""Build the concrete tensors: a fan of light rays, the sky pixel grid and the path of the background source.

Rays: units G = c = M = 1 (lengths in GM/c²). Sky: angles in units of the Einstein radius θ_E.
"""

import torch


def ray_fan(b_min: float, b_max: float, n: int, device: torch.device) -> torch.Tensor:
    """Impact parameters b (B,) and initial states y0 (B, 2): u = 0 (at infinity), du/dφ = 1/b."""
    b = torch.linspace(b_min, b_max, n, dtype=torch.float64, device=device)
    return b, torch.stack([torch.zeros_like(b), 1 / b], dim=1)


def sky_grid(n: int, half_width: float, device: torch.device) -> torch.Tensor:
    """Angles θ_{pi} of an n × n pixel grid, shape (n, n, 2), offset by half a pixel so no pixel sits on the lens."""
    s = torch.linspace(-half_width, half_width, n + 1, device=device)
    s = 0.5 * (s[1:] + s[:-1])
    X, Y = torch.meshgrid(s, s, indexing="xy")
    return torch.stack([X, Y], dim=-1)


def source_path(n: int, impact: float, half_length: float, device: torch.device) -> torch.Tensor:
    """Source centre (F, 2) moving along x at height `impact` behind the lens."""
    x = torch.linspace(-half_length, half_length, n, device=device)
    return torch.stack([x, torch.full_like(x, impact)], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/gravitational_lensing/solve.py
"""Trace the null geodesics with RK4 in φ, and lens the moving source on the whole pixel grid at once."""

import math

import torch

from problems._common.integrators import integrate

from .equation import binet_rhs, lens_map, source_brightness


def trace_rays(y0: torch.Tensor, M: float, dphi: float = 0.004, phi_max: float = 4 * math.pi) -> dict:
    """Integrate u(φ) for every ray. A ray is done when it escapes (u < 0) or falls in (u > 1/(2M): horizon)."""
    n = int(round(phi_max / dphi))
    traj = integrate(lambda y, phi: binet_rhs(y, M), y0, dphi, n)  # (T, B, 2)
    u = traj[:, :, 0]
    phi = torch.arange(n + 1, dtype=u.dtype, device=u.device) * dphi
    escaped = (u < 0) & (phi[:, None] > 0.5)
    captured = u > 1 / (2 * M)
    done = torch.cummax((escaped | captured).int(), dim=0).values.bool()
    return {"phi": phi, "u": u.masked_fill(done, float("nan")), "escaped": escaped.any(0), "raw_u": u}


def deflection_angles(rays: dict) -> torch.Tensor:
    """α = φ_out − π, with φ_out where u returns to zero (linear interpolation); NaN for captured rays, (B,)."""
    u, phi = rays["raw_u"], rays["phi"]
    alpha = torch.full((u.shape[1],), float("nan"), dtype=u.dtype)
    for b in range(u.shape[1]):
        crossing = torch.nonzero((u[:-1, b] > 0) & (u[1:, b] <= 0) & (phi[1:] > 0.5)).flatten()
        if len(crossing) and rays["escaped"][b]:
            k = int(crossing[0])
            f = u[k, b] / (u[k, b] - u[k + 1, b])
            alpha[b] = phi[k] + f * (phi[k + 1] - phi[k]) - math.pi
    return alpha


def ray_paths(rays: dict) -> torch.Tensor:
    """Cartesian points (T, B, 2) of each ray, coming in from the left: x = −cos φ / u, y = sin φ / u."""
    r = 1 / rays["u"]
    phi = rays["phi"][:, None]
    return torch.stack([-r * torch.cos(phi), r * torch.sin(phi)], dim=-1)


def lensed_images(theta: torch.Tensor, centres: torch.Tensor, radius: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Lensed and unlensed images (F, n, n) for each source position (F, 2), all frames in one broadcast."""
    beta = lens_map(theta)  # (n, n, 2): where each pixel looks in the source plane
    lensed = source_brightness(beta[None], centres[:, None, None, :], radius)  # (F, n, n)
    plain = source_brightness(theta[None], centres[:, None, None, :], radius)  # (F, n, n) without the lens
    return lensed, plain


def magnification(lensed: torch.Tensor, plain: torch.Tensor) -> torch.Tensor:
    """Total flux ratio per frame (pixel areas cancel), (F,)."""
    return lensed.sum(dim=(-2, -1)) / plain.sum(dim=(-2, -1))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/gravitational_lensing/plot.py
"""Light rays bending around the black hole, the lensed sky as a source passes behind it, and its light curve."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import point_magnification


def animate(rays: dict, b: torch.Tensor, alpha: torch.Tensor, M: float, sky: dict, path: Path) -> Path:
    pts = rays["paths"].cpu().numpy()  # (T, B, 2)
    b, alpha = b.cpu().numpy(), alpha.cpu().numpy()
    lensed = sky["lensed"].cpu().numpy()  # (F, n, n)
    centres = sky["centres"].cpu().numpy()  # (F, 2)
    mu = sky["mu"].cpu().numpy()
    half = sky["half_width"]
    F = len(centres)

    fig = plt.figure(figsize=(8, 4.5))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.1, 1], height_ratios=[1.2, 1])
    ax_r = fig.add_subplot(gs[0, 0])
    ax_l = fig.add_subplot(gs[1, 0])
    ax_i = fig.add_subplot(gs[:, 1])

    # --- null geodesics past the hole ---
    ax_r.set_xlim(-25, 25)
    ax_r.set_ylim(-9, 16)
    ax_r.set_aspect("equal", adjustable="box")
    ax_r.add_patch(plt.Circle((0, 0), 2 * M, color="black", zorder=4))
    ax_r.add_patch(plt.Circle((0, 0), 3 * M, fill=False, ls=":", color="#f59e0b", lw=1.2))
    for k in range(pts.shape[1]):
        captured = not np.isfinite(alpha[k])
        ax_r.plot(pts[:, k, 0], pts[:, k, 1], lw=1.2, color="#6b7280" if captured else ACCENT[k % len(ACCENT)])
    ax_r.set_title(f"null geodesics; b < √27 M = {np.sqrt(27) * M:.2f} M are captured", fontsize=8)
    ax_r.set_xlabel("x (GM/c²)", fontsize=8)
    ax_r.tick_params(labelsize=7)

    # --- light curve ---
    x_src = centres[:, 0]
    u = np.linalg.norm(centres, axis=-1)
    ax_l.semilogy(x_src, point_magnification(torch.tensor(u)).numpy(), color="#9ca3af", lw=1, label="point source")
    curve = ax_l.semilogy([], [], color=ACCENT[0], lw=2, label="this source (pixel sum)")[0]
    ax_l.set_xlim(x_src[0], x_src[-1])
    ax_l.set_ylim(0.9, max(mu.max(), 3) * 1.5)
    ax_l.set_xlabel("source position β_x (θ_E)", fontsize=8)
    ax_l.set_ylabel("magnification μ", fontsize=8)
    ax_l.tick_params(labelsize=7)
    ax_l.legend(fontsize=7, loc="upper left")

    # --- the lensed sky ---
    img = ax_i.imshow(lensed[0], origin="lower", extent=(-half, half, -half, half), cmap="magma", vmin=0, vmax=1)
    ring = np.linspace(0, 2 * np.pi, 200)
    ax_i.plot(np.cos(ring), np.sin(ring), ls="--", lw=0.8, color="#93c5fd")
    src = ax_i.plot([], [], "+", color="#93c5fd", ms=10, mew=1.5)[0]
    ax_i.plot([0], [0], "o", color="white", ms=3)
    ax_i.set_xlabel("θ_x (θ_E)", fontsize=8)
    ax_i.set_ylabel("θ_y (θ_E)", fontsize=8)
    ax_i.tick_params(labelsize=7)
    ax_i.grid(False)
    title = ax_i.set_title("", fontsize=9)

    def update(k: int):
        img.set_data(lensed[k])
        src.set_data([centres[k, 0]], [centres[k, 1]])
        curve.set_data(x_src[: k + 1], mu[: k + 1])
        title.set_text(f"lensed sky   μ = {mu[k]:.2f}   (+ true source position)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(F, 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'b_min': 4.0,
    'b_max': 14.0,
    'n_rays': 9,
    'impact': 0.15,
    'source_radius': 0.12,
    'n_pixels': 200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.gravitational_lensing")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.gravitational_lensing.main")
problem_dir = Path("problems/astro/gravitational_lensing")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
